In [12]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Any
from openai import OpenAI

client = OpenAI()

def embed_text(text: str) -> np.ndarray:
    response = client.embeddings.create(
        input=text,
        model="text-embedding-3-large"
    )

    embedding = response.data[0].embedding    
    return embedding

def top_k_candidates(nfr_emb: np.ndarray, story_embs: np.ndarray, k: int):
    sims = cosine_similarity(nfr_emb.reshape(1, -1), story_embs).flatten()
    top_idx = np.argsort(sims)[::-1][:k]
    return top_idx, sims[top_idx]

def make_context(nfr_emb: np.ndarray, story_emb: np.ndarray, sim_score: float, meta_n: np.ndarray = None, meta_s: np.ndarray = None):
    parts = [nfr_emb, story_emb, np.array([sim_score])]
    if meta_n is not None: 
        parts.append(meta_n)
    if meta_s is not None:
        parts.append(meta_s)
    ctx = np.concatenate(parts)
    norm = np.linalg.norm(ctx) + 1e-9
    return ctx / norm

class LinUCB:
    def __init___(self, dim: int, alpha: float = 1.0):
        self.d = dim
        self.alpha = alpha
        self.A = np.eye(dim) * 1.0 #regularization
        self.b = np.zeros(dim)

    def predict(self, x: np.ndarray):
        theta = np.linalg.solve(self.A, self.b)
        mean = float(theta.dot(x))
        Ainv = np.linalg.inv(self.A)
        var = float(x.dot(Ainv).dot(x))
        ucb = mean + self.alpha * np.sqrt(var)
        return ucb, mean

    def update(self, x: np.ndarray, reward: float):
        x = x.reshape(-1, 1)
        self.A += x.dot(x.T)
        self.b += (reward * x).flatten()

#-------------------------
# Pipeline
#-------------------------
def run_demo(nfr_texts: List[str], story_texts: List[str], ground_truth: Dict[int, set], K=5, rounds=200):
    # 1) Embeddings
    nfr_embs = np.vstack([embed_text(t) for t in nfr_texts])
    story_embs = np.vstack([embed_text(t) for t in story_texts])

    # 2) Candidate precompute
    all_sim = cosine_similarity(nfr_embs, story_embs)
    candidates = {i: list(np.argsort(all_sim[i])[::-1][:K]) for i in range(len(nfr_texts))}

    # 3) Meta
    meta_dim = 4
    meta_nfr = np.zeros((len(nfr_text), meta_dim))
    meta_story = np.zeros = np.zeros((len(story_texts), meta_dim))
    # TODO: Maybe extract the attributes from an LLM, need to think about it, how to handle this

    # 4) Agent init
    emb_dim = nfr_embs.shape[1]
    ctx_dim = emb_dim*2 + 1 + meta_dim*2
    agent = LinUCB(ctx_dim, alpha=0.8)

    stats = {"proposed": 0, "accepeted": 0, "precision_history": []}

    for t in range(rounds):
        
        # will randomly choose an NFR to evaluate
        nfr_idx = np.random.randint(len(nfr_texts))
        cand_list = candidates[nfr_idx]

        # will select best candidate by agent UCB
        best_ucb = -np.inf
        best_sid = None
        best_ctx = None
        for sid in cand_list:
            sim = float(all_sim[nfr_idx, sid])
            ctx = make_context(nfr_embs[nfr_idx], story_embs[sid], sim, meta_nfr[nfr_idx], meta_story[sid])
            ucb, mean = agent.predict(ctx)
            if ucb > best_ucb:
                best_ucb = ucb
                best_sid = sid
                best_ctx = ctx

        # Will decide to propose or not (threshold policy)
        threshold = 0.0 # TODO need to decide what threshold I want to choose
        if best_ucb > threshold:
            # Currently: simualated user feedback, in production the feedback of the user will be taken
            accepted = 1 if best_sid in ground_truth.get(nfr_idx, set()) else 0
            reward = 1.0 if accepted else -1.0
            agent.update(best_ctx, reward)
            stats["proposed"] += 1
            stats["accepted"] += accepted

        # record precision
        precision = stats["accepted"] / stats["proposed"] if stats["proposed"] > 0 else 0.0
        stats["precision_history"].append(precision)

    return stats

### Missing: Sample NFRs/User Story texts, ground truth map
         
